# AN-RA iterate500 TPU Training
Dedicated Colab TPU path for the 500M-class frontier model using PyTorch/XLA. This notebook is separate from the T4 notebook on purpose.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['PJRT_DEVICE'] = 'TPU'
print('PJRT_DEVICE=', os.environ['PJRT_DEVICE'])

# Install the TPU runtime stack first. Do not use the T4/CUDA constraints here.
!pip install -q torch 'torch_xla[tpu]' -f https://storage.googleapis.com/libtpu-releases/index.html

In [ ]:
REPO = '/content/An-Ra-the-new-AGI'
BRANCH = 'iterate500'
![ -d "$REPO/.git" ] || git clone --branch "$BRANCH" --single-branch https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git "$REPO"
%cd $REPO
!git fetch origin "$BRANCH"
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"

# Install non-torch dependencies and AN-RA itself without replacing torch_xla's torch.
!pip install -q 'datasets==3.2.0' 'transformers>=4.40.0' 'tokenizers>=0.19.0' 'tqdm>=4.66.0' 'numpy>=1.26.0' 'pyyaml>=6.0.0' 'psutil>=5.9.0' 'scipy>=1.12.0' 'sympy>=1.12.0'
![ -d /content/thirdeye/.git ] || git clone --depth 1 https://github.com/dhurv0045com-spec/thirdeye.git /content/thirdeye
!pip install -q -e /content/thirdeye
!pip install -q --no-deps -e "$REPO"

!python scripts/colab_tpu_bootstrap.py --repo "$REPO" --drive-root /content/drive/MyDrive/AnRa --install-repo

In [ ]:
%cd /content/An-Ra-the-new-AGI
from pathlib import Path
DATA_PROFILE = 'tpu'
FORCE_DATA_REBUILD = False
data_ready = Path('training_data/base_corpus.txt').exists() and Path('training_data/reasoning.jsonl').exists()
if FORCE_DATA_REBUILD or not data_ready:
    !python scripts/download_training_data.py --profile $DATA_PROFILE --prepare-corpus
else:
    print(f'Data already prepared for this runtime. Using profile={DATA_PROFILE}.')
!python -m data.causal_corpus
!python scripts/evaluate_with_thirdeye.py --profile quick --without-model

In [ ]:
%cd /content/An-Ra-the-new-AGI
import os
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ.setdefault('ANRA_CHECKPOINT_EVERY_MIN', '25')
# Set to '0' only for a baseline if ThirdEye telemetry overhead is too high.
os.environ.setdefault('ANRA_THIRDEYE_INTELLIGENCE', '1')

SESSION_MINUTES = 180
!python scripts/build_brain_tpu.py --data_path training_data/anra_training.txt --checkpoint_path anra_frontier_500m.pt --model-size frontier --batch_size 1 --grad_accum_steps 16 --optimizer adafactor --max_minutes $SESSION_MINUTES --log_every 1
!python scripts/evaluate_with_thirdeye.py --profile quick --without-model